# Teachable Machine Projects (B1, B2, B3, B4)
Note: Since I cannot use your webcam or microphone to collect data, you must complete the data collection and model export steps (Task 1-3 for each section) yourself in the browser using Teachable Machine. Once you have exported the models and placed them in the appropriate directories, you can run the cells below.

## Task B1: Image Project - Gesture / Object Classifier

In [9]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install cv2 
pip install numpy
pip install os
pip install pyautogui

SyntaxError: invalid syntax (3687420004.py, line 1)

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import os
import pyautogui # For B1.5 gesture control

# Ensure output directory exists
os.makedirs('DL_PR4B/output', exist_ok=True)

# (a) Load model
try:
    image_model = tf.keras.models.load_model('DL_PR4B/models/image/keras_model.h5', compile=False)
    
    # (b) Load labels
    with open('DL_PR4B/models/image/labels.txt', 'r') as f:
        image_labels = [line.strip() for line in f.readlines()]
        
    print("Image model loaded successfully.")
except Exception as e:
    print(f"Error loading image model: {e}")
    print("Please ensure keras_model.h5 and labels.txt are in DL_PR4B/models/image/")

In [ ]:
# B1.4 Deploy: image_classifier.py & B1.5 Build something useful (Gesture volume control)
def run_image_classifier():
    cap = cv2.VideoCapture(0)
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        # (d) Preprocess: resize to 224x224, normalize to [-1,1]
        img = cv2.resize(frame, (224, 224))
        img = np.asarray(img, dtype=np.float32)
        img = (img / 127.5) - 1.0
        img = np.expand_dims(img, axis=0)
        
        # (e) Predict
        prediction = image_model.predict(img, verbose=0)
        class_idx = np.argmax(prediction)
        confidence = prediction[0][class_idx]
        class_name = image_labels[class_idx]
        
        # B1.5 Gesture control (example: thumbs up -> volumeup, thumbs down -> volumedown)
        if confidence > 0.8:
            if "thumbs up" in class_name.lower():
                pyautogui.press('volumeup')
            elif "thumbs down" in class_name.lower():
                pyautogui.press('volumedown')
        
        # (f) Overlay prediction
        cv2.putText(frame, f"{class_name}: {confidence:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('Image Classifier', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cv2.imwrite('DL_PR4B/output/b1_image_result.jpg', frame)
            break
            
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to run
# run_image_classifier()

## Task B2: Audio Project - Sound / Command Classifier

In [ ]:
import sounddevice as sd
import librosa # Need librosa for mel spectrogram, or can use TFLite support

# Load audio model
try:
    audio_model = tf.keras.models.load_model('DL_PR4B/models/audio/keras_model.h5', compile=False)
    with open('DL_PR4B/models/audio/labels.txt', 'r') as f:
        audio_labels = [line.strip() for line in f.readlines()]
    print("Audio model loaded successfully.")
except Exception as e:
    print(f"Error loading audio model: {e}")

In [ ]:
# B2.4 Deploy audio classifier
def run_audio_classifier():
    # NOTE: Teachable Machine audio preprocessing is complex.
    # This is a simplified placeholder structure. 
    # For full correctness, using tm-audio-snippet or tf.lite support is recommended.
    duration = 1  # seconds
    fs = 44100  # Sample rate
    clap_count = 0

    while True:
        print("Recording...")
        audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
        sd.wait()
        
        # Preprocessing (Simplified/Placeholder: Requires proper mel-spectrogram conversion)
        # audio_padded = ... convert to (1, 43, 232, 1) spectrogram ...
        # prediction = audio_model.predict(audio_padded)
        # class_idx = np.argmax(prediction)
        # ...
        print("Audio processed (requires proper preprocessing pipeline implementation).")
        
        # B2.5 Build something useful (Clap counter example)
        # if audio_labels[class_idx] == 'clapping' and confidence > 0.85:
        #     clap_count += 1
        #     print(f"\rClap Count: {clap_count}", end="")
        
        break # break after 1 iteration for safety in notebook

# run_audio_classifier()

## Task B3: Pose Project - Body Pose Classifier

In [ ]:
import tensorflow_hub as hub

try:
    # (a) Load MoveNet
    movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
    movenet_model = movenet.signatures['serving_default']
    
    # (b) Load TM Pose Model
    pose_model = tf.keras.models.load_model('DL_PR4B/models/pose/keras_model.h5', compile=False)
    with open('DL_PR4B/models/pose/labels.txt', 'r') as f:
        pose_labels = [line.strip() for line in f.readlines()]
        
    print("Pose models loaded successfully.")
except Exception as e:
    print(f"Error loading pose models: {e}")

In [ ]:
def run_pose_classifier():
    cap = cv2.VideoCapture(0)
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        # (d) Run MoveNet
        # Resize to 192x192 for MoveNet Lightning
        img = frame.copy()
        img = tf.image.resize_with_pad(np.expand_dims(img, axis=0), 192, 192)
        input_image = tf.cast(img, dtype=tf.int32)
        
        outputs = movenet_model(input_image)
        keypoints = outputs['output_0'].numpy()[0, 0, :, :]
        
        # Flatten to (1,34) as expected by TM Pose Model
        flat_keypoints = keypoints[:, :2].flatten().reshape(1, 34)
        
        # Predict pose
        prediction = pose_model.predict(flat_keypoints, verbose=0)
        class_idx = np.argmax(prediction)
        confidence = prediction[0][class_idx]
        class_name = pose_labels[class_idx]
        
        # (e) Draw keypoints and (f) Overlay prediction
        y, x, _ = frame.shape
        for kp in keypoints:
            ky, kx, kp_conf = kp
            if kp_conf > 0.3:
                cv2.circle(frame, (int(kx * x), int(ky * y)), 4, (0, 255, 0), -1)
                
        cv2.putText(frame, f"{class_name}: {confidence:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        
        cv2.imshow('Pose Classifier', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            # (g) Save output (for simplicity saving single image here)
            cv2.imwrite('DL_PR4B/output/b3_pose_result.jpg', frame)
            break
            
    cap.release()
    cv2.destroyAllWindows()

# run_pose_classifier()

## Task B4: Combined Application - Multi-Modal Demo

In [ ]:
'''
B4.1 Design
(a) What the application does: Integrates Image, Audio, and Pose classification.
(b) Models loaded: MobileNet (Image), AudioNet (Audio), PoseNet/MoveNet (Pose)
(c) Interaction Flow: Webcam shows real-time image and pose classification, while a background thread monitors audio. Dashboard displays all statuses.
'''
import threading
import time

audio_status = "Silence"

def audio_thread_func():
    global audio_status
    while True:
        # Simulate audio classification every 2 seconds
        time.sleep(2)
        audio_status = "Checking..."
        # sd.rec(...)
        # audio_status = ...

def run_combined_app():
    # Start audio thread
    t = threading.Thread(target=audio_thread_func, daemon=True)
    t.start()
    
    cap = cv2.VideoCapture(0)
    # Setup VideoWriter for b4_combined.mp4 ...
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        # Run Image Classifier
        # ...
        
        # Run Pose Classifier
        # ...
        
        # Dashboard Overlay
        cv2.putText(frame, f"Audio: {audio_status}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        cv2.imshow('Combined Dashboard', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

# run_combined_app()

In [ ]:
# B4.4 Model Introspection
print("--- Image Model ---")
if 'image_model' in globals():
    image_model.summary()
    print("Input shape:", image_model.input_shape)
    print("Output shape:", image_model.output_shape)

print("\n--- Pose Model ---")
if 'pose_model' in globals():
    pose_model.summary()
    print("Input shape:", pose_model.input_shape)
    print("Output shape:", pose_model.output_shape)

'''
Comment: Teachable Machine's image model is typically MobileNetV2 truncated at an intermediate layer 
+ a small Dense head — only the Dense head weights are your custom-trained parameters.
'''